# 01 · File formats
From class: CSV, TSV, XLSX, XML, JSON (and Parquet, the columnar format data lakes use). Same data, different trade-offs: size, speed, whether types survive the round trip.

In [ ]:
import sys
sys.path.insert(0, "..")          # lets the notebook import common.py from python_practice/
from common import *              # load_operations, check, OUT, ...
import pandas as pd

df = load_operations()            # reads the dashboard API if it is running, else data_samples/operations_clean.csv

In [ ]:
sample = df.head(2000).copy()
sample["record_date"] = sample["record_date"].dt.strftime("%Y-%m-%d")   # plain text dates, so every format gets the same input
paths = {
    "csv": OUT / "demo.csv", "tsv": OUT / "demo.tsv", "json": OUT / "demo.json",
    "xlsx": OUT / "demo.xlsx", "xml": OUT / "demo.xml", "parquet": OUT / "demo.parquet",
}
sample.to_csv(paths["csv"], index=False)
sample.to_csv(paths["tsv"], index=False, sep="\t")
sample.to_json(paths["json"], orient="records", indent=1)
sample.to_excel(paths["xlsx"], index=False)
sample.to_xml(paths["xml"], index=False, parser="etree")
sample.to_parquet(paths["parquet"], index=False)
sizes = {k: p.stat().st_size for k, p in paths.items()}
pd.Series(sizes, name="bytes").sort_values()

## Read them back and compare
What type does `record_date` come back as? That is the real difference between a text format and a typed one.

In [ ]:
readers = {"csv": pd.read_csv, "tsv": lambda p: pd.read_csv(p, sep="\t"), "json": pd.read_json,
           "xlsx": pd.read_excel, "xml": pd.read_xml, "parquet": pd.read_parquet}
rows = []
for name, read in readers.items():
    back = read(paths[name])
    rows.append({"format": name, "rows": len(back), "record_date dtype": str(back["record_date"].dtype), "revenue dtype": str(back["revenue"].dtype)})
pd.DataFrame(rows)

## Your turn
Which format produced the **smallest** file? Put its name (like `"csv"`) in the variable. Then think: would you pick it for a file you email to a colleague?

In [ ]:
smallest = None

In [ ]:
check("smallest format", smallest == min(sizes, key=sizes.get), "look at the sizes table above, or use min(sizes, key=sizes.get)")

**Discuss:** CSV/TSV are human-readable but untyped. JSON/XML carry structure. XLSX is for people. Parquet is compressed and typed, the choice for analytics at scale (Spark and DuckDB read it directly).